## 05 Reporting

Pull in all the results from notebooks 03 and 04, the metrics, confusion matrices, disagreement outputs, and LIME/SHAP tables.

We build a clean performance table with the F1, precision, recall and accuracy numbers for all four model and language combinations.

From there it generates an F1 bar chart comparing XLM-R and AfriBERTa side by side for both languages, with the scores labelled on each bar. It also takes the four individual confusion matrix images and combines them into one 2x2 grid so all models can be compared at a glance.

After that it loads and displays all the LIME and SHAP tables from the XAI analysis, showing which words each model focused on for the disagreement cases.

Finally it prints a clean summary of all the key results and the main finding in one block, and saves everything to an outputs/report folder on your Drive.


In [ ]:

!pip -q install matplotlib seaborn pandas

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# set a clean visual style for all plots in this notebook
sns.set_theme(style="whitegrid")

PROJECT_PATH = "/content/drive/Shareddrives/Cos760"

# define all the directories we'll be reading from and writing to
METRICS_DIR  = os.path.join(PROJECT_PATH, "outputs", "metrics")
ANALYSIS_DIR = os.path.join(PROJECT_PATH, "outputs", "analysis")
XAI_DIR      = os.path.join(ANALYSIS_DIR, "xai")
TABLE_DIR    = os.path.join(XAI_DIR, "tables")
REPORT_DIR   = os.path.join(PROJECT_PATH, "outputs", "report")

# create the report output folder if it doesn't already exist
os.makedirs(REPORT_DIR, exist_ok=True)
print("Report output directory:", REPORT_DIR)

In [ ]:
# load the metrics CSV produced by notebook 04
# if the file isn't found, fall back to hardcoded values so the notebook doesn't crash
metrics_path = os.path.join(METRICS_DIR, "analysis_model_metrics.csv")

if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
else:
    metrics_df = pd.DataFrame([
        {"language": "hausa",       "model_name": "xlmr_hausa",           "f1_weighted": 0.7411, "precision_weighted": 0.7425, "recall_weighted": 0.7407, "accuracy": 0.7407},
        {"language": "hausa",       "model_name": "afriberta_hausa",       "f1_weighted": 0.7942, "precision_weighted": 0.7939, "recall_weighted": 0.7950, "accuracy": 0.7950},
        {"language": "kinyarwanda", "model_name": "xlmr_kinyarwanda",      "f1_weighted": 0.5661, "precision_weighted": 0.5839, "recall_weighted": 0.5750, "accuracy": 0.5750},
        {"language": "kinyarwanda", "model_name": "afriberta_kinyarwanda", "f1_weighted": 0.6193, "precision_weighted": 0.6192, "recall_weighted": 0.6199, "accuracy": 0.6199},
    ])

display(metrics_df)
print("Metrics loaded.")


metrics_df["Model"]    = metrics_df["model_name"].str.replace("_hausa", "").str.replace("_kinyarwanda", "").str.upper()
metrics_df["Language"] = metrics_df["language"].str.capitalize()

report_table = metrics_df[["Language", "Model", "f1_weighted", "precision_weighted", "recall_weighted", "accuracy"]].copy()
report_table.columns = ["Language", "Model", "F1 (Weighted)", "Precision", "Recall", "Accuracy"]
report_table = report_table.sort_values(["Language", "Model"]).reset_index(drop=True)

display(report_table)

# saving the table as a CSV for the report
report_table.to_csv(os.path.join(REPORT_DIR, "performance_summary_table.csv"), index=False)
print("Saved performance summary table.")

In [ ]:
# creatnig a side by side bar chart comparing weighted F1 for both languages
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for i, lang in enumerate(["hausa", "kinyarwanda"]):
    lang_df = metrics_df[metrics_df["language"] == lang]

    sns.barplot(
        data=lang_df,
        x="Model",
        y="f1_weighted",
        palette=["#378ADD", "#1D9E75"],
        ax=axes[i]
    )

    axes[i].set_title(f"{lang.capitalize()} — Weighted F1 Score", fontsize=13)
    axes[i].set_xlabel("Model")
    axes[i].set_ylabel("Weighted F1")
    axes[i].set_ylim(0, 1)

    # labelling each bar with its exact score
    for bar in axes[i].patches:
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=11
        )

plt.suptitle("XLM-R vs AfriBERTa: Weighted F1 Score by Language", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "f1_comparison_chart.png"), dpi=300)
plt.show()
print("Saved F1 comparison chart.")

In [ ]:
# loading the four individual confusion matrix images and combine into a 2x2 grid
confusion_files = {
    "XLM-R Hausa":           os.path.join(ANALYSIS_DIR, "confusion_matrix_xlmr_hausa.png"),
    "AfriBERTa Hausa":       os.path.join(ANALYSIS_DIR, "confusion_matrix_afriberta_hausa.png"),
    "XLM-R Kinyarwanda":     os.path.join(ANALYSIS_DIR, "confusion_matrix_xlmr_kinyarwanda.png"),
    "AfriBERTa Kinyarwanda": os.path.join(ANALYSIS_DIR, "confusion_matrix_afriberta_kinyarwanda.png"),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (title, path) in enumerate(confusion_files.items()):
    if os.path.exists(path):
        img = plt.imread(path)
        axes[i].imshow(img)
        axes[i].set_title(title, fontsize=12)
        axes[i].axis("off")
    else:
        axes[i].set_title(f"{title} — NOT FOUND", fontsize=12)
        axes[i].axis("off")

plt.suptitle("Confusion Matrices: All Models and Languages", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, "confusion_matrices_combined.png"), dpi=300)
plt.show()
print("Saved combined confusion matrices.")

In [ ]:
# loading and display the disagreement outcomes plot and summary table
disagreement_plot = os.path.join(ANALYSIS_DIR, "disagreement_outcomes.png")
if os.path.exists(disagreement_plot):
    display(Image(disagreement_plot))
else:
    print("Disagreement outcomes plot not found.")

disagreement_summary_path = os.path.join(METRICS_DIR, "disagreement_summary.csv")
if os.path.exists(disagreement_summary_path):
    disagreement_summary = pd.read_csv(disagreement_summary_path)
    display(disagreement_summary)

In [ ]:
# loading and displaying all LIME and SHAP output tables from the XAI analysis
xai_case_path  = os.path.join(TABLE_DIR, "xai_case_level_summary.csv")
lime_shap_path = os.path.join(TABLE_DIR, "lime_shap_comparison_by_case.csv")
lime_top_path  = os.path.join(TABLE_DIR, "lime_top_features_summary.csv")
shap_top_path  = os.path.join(TABLE_DIR, "shap_top_features_summary.csv")

if os.path.exists(xai_case_path):
    display(pd.read_csv(xai_case_path))

if os.path.exists(lime_shap_path):
    display(pd.read_csv(lime_shap_path))

if os.path.exists(lime_top_path):
    display(pd.read_csv(lime_top_path))

if os.path.exists(shap_top_path):
    display(pd.read_csv(shap_top_path))

In [ ]:
# print a clean summary of all key results and the main finding
print("=" * 60)
print("FINAL RESULTS SUMMARY — GROUP 23")
print("=" * 60)
print()
print("Research Question:")
print("To what extent do XLM-R and AfriBERTa differ in sentiment")
print("classification, and what do LIME and SHAP reveal about why?")
print()
print("Performance Results:")
print()
for _, row in report_table.iterrows():
    print(f"  {row['Language']:12} | {row['Model']:10} | F1: {row['F1 (Weighted)']:.4f} | Accuracy: {row['Accuracy']:.4f}")
print()
print("Key Finding:")
print("AfriBERTa outperforms XLM-R on both languages.")
print("The gap is larger on Hausa (0.058 F1) than Kinyarwanda")
print("(0.053 F1), consistent with AfriBERTa pretraining on Hausa.")
print()
print("All report outputs saved to:", REPORT_DIR)